# AfterFix: reproduce the separate study results on CPU

**Nitish Songara, Independent researcher.** Original code and synthetic benchmark: Apache 2.0. Manuscript: CC BY 4.0 OR Apache 2.0; third-party exclusions are in the package license scope.

This notebook regrades 219 saved model attempts and replays 941 feedback records. It keeps the original 12B NF4 study, corrected H08, matched v3 31B/12B study, and post hoc CPU diagnostic separate. It does not generate model outputs or reproduce GPU timings.

Input: [versioned CPU package](https://www.kaggle.com/datasets/nitishsongara/afterfix-cpu-reproduction). Use accelerator **None** and Internet **off**. Kaggle may expand the uploaded ZIP into nested folders; the code accepts that layout only after checking the exact package manifest and every file hash. Earlier review wording inside the sealed input is historical and its bytes are preserved.

The package requires Python 3.12 or newer and standard sqlite3. Any runtime discrepancy fails visibly; expected historical outputs are never rewritten.

V6 permits CRLF/LF differences in regenerated case-review documents and reports byte/text equality separately. V1 of this notebook failed that document-only comparison after all grades and feedback passed. Historical inputs and scores remain unchanged.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import uuid
import zipfile

assert sys.version_info >= (3, 12), "Python 3.12 or newer is required"
CPU_ARCHIVE_SHA256 = "9cc3170e3a6cd90814b74a5848ef2339aa8f90a4ddf13649982b9b5230c01c69"
MANIFEST_SHA256 = "ba814886417c483cb0e1f736348830a8ad0bd37f28bd60347fad5bbb540713a3"
PACKAGE_NAME = "afterfix-cpu-v6"
workspace = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
work = workspace / ('afterfix-cpu-reproduction-' + uuid.uuid4().hex[:12])
work.mkdir(exist_ok=False)
explicit = os.environ.get('AFTERFIX_CPU_PACKAGE')
search_root = Path('/kaggle/input')
if explicit:
    package = Path(explicit).resolve()
else:
    candidates = [p.parent for p in search_root.rglob('reproduce.py')
                  if (p.parent / 'MANIFEST.json').is_file()
                  and hashlib.sha256((p.parent / 'MANIFEST.json').read_bytes()).hexdigest() == MANIFEST_SHA256]
    if not candidates:
        archives = list(search_root.rglob(PACKAGE_NAME + '.zip'))
        assert len(archives) == 1, 'Attach the exact CPU reproduction dataset'
        assert hashlib.sha256(archives[0].read_bytes()).hexdigest() == CPU_ARCHIVE_SHA256
        extract = work / 'input'
        with zipfile.ZipFile(archives[0]) as z:
            for name in z.namelist():
                assert name.startswith(PACKAGE_NAME + '/')
                assert (extract / name).resolve().is_relative_to(extract.resolve())
            assert z.testzip() is None
            z.extractall(extract)
        candidates = [extract / PACKAGE_NAME]
    assert len(candidates) == 1, 'Expected exactly one verified CPU package'
    package = candidates[0].resolve()
assert hashlib.sha256((package / 'MANIFEST.json').read_bytes()).hexdigest() == MANIFEST_SHA256
manifest = json.loads((package / 'MANIFEST.json').read_text(encoding='utf-8'))
for name, expected in manifest['files'].items():
    path = (package / name).resolve()
    assert path.is_relative_to(package) and path.is_file(), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
print('Verified input:', package)
print('Output:', work)


In [ ]:
result = subprocess.run([sys.executable, '-I', '-B', str(package / 'reproduce.py'),
                         '--output', str(work / 'rebuilt')], cwd=package, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, 'Replay failed; retain outputs and investigate without replacing history'
verification = json.loads((work / 'rebuilt/verification.json').read_text(encoding='utf-8'))
assert verification['status'] == 'passed'
assert sum(verification['studies'][s]['exact_grade_matches'] for s in ['original','h08','v3']) == 219
assert sum(verification['studies'][s]['exact_feedback_replays'] for s in ['original','h08','v3']) == 941
print(json.dumps(verification, indent=2))
tables = (work / 'rebuilt/PAPER_TABLES.md').read_text(encoding='utf-8')
try:
    from IPython.display import Markdown, display
    display(Markdown(tables))
except ImportError:
    print(tables)


## Interpretation

These are regenerated grades from saved responses. Original and corrected H08 scores are never combined. V3 uses different budgets and runtime from the original and reuses inspected cases. It is not a fresh holdout or a causal test of parameter count. Checklist and trace access show no consistent advantage.

The later scope-adaptation diagnostic solves 24/24 cases. It was developed after inspecting outcomes and exposes shared task structure; it is not a new recovery algorithm. Stable execution alone does not establish correct recovery. The package documents boundary probes and AI-assisted internal review, which is not independent domain validation.

This CPU notebook is a reproducibility artifact. Publishing or running it does not submit a competition Writeup.
